In [ ]:
from pathlib import Path
from zipfile import ZipFile

from open_video_summary.utils.config import DataPaths
from open_video_summary.parsers.video import VideoLoader, VideoDumper
from open_video_summary.core.segmenter.video_segmenter import WordVideoSegmenter

# Unzipping sample dataset video files

In [ ]:
zip_path = (Path(DataPaths.RAW_PATH) / "bebe_real.zip").as_posix()
extract_path = (Path(DataPaths.RAW_PATH)).as_posix()

with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

processed_data = Path(DataPaths.RAW_PATH) / "bebe_real" / "bebe_real.json"
processed_data.rename(Path(DataPaths.PROCESSED_PATH) / "bebe_real.json")

# Loading data, segmenting videos, and saving output file

In [ ]:

datasets = [
    "bebe_real",
    "bolsonaro_trump",
    "brumadinho",
    "catedral_notre_dame",
    "google_huawei",
]
for dataset in datasets:
    dataset_videos_path = (Path(DataPaths.RAW_PATH) / dataset).as_posix()
    dataset_json_path = (Path(DataPaths.PROCESSED_PATH) / f"{dataset}.json").as_posix()

    video_segmenter = WordVideoSegmenter(
        whisper_model="medium",
        min_segment_length=5,
        max_segment_length=120,
    )

    videos = [
        raw_video for raw_video in VideoLoader.load_videos_from_directory(dataset_videos_path)
    ]

    videos = video_segmenter.create_videos_segments(videos)
    VideoDumper.dump_videos_to_json(videos, dataset_json_path)

# Dumping segments to CSV file for analysis

In [ ]:

datasets = [
    "bebe_real",
    "bolsonaro_trump",
    "brumadinho",
    "catedral_notre_dame",
    "google_huawei",
]

data = []
for dataset in datasets:
    dataset_videos_path = (Path(DataPaths.RAW_PATH) / dataset).as_posix()
    dataset_json_path = (Path(DataPaths.PROCESSED_PATH) / f"{dataset}.json").as_posix()

    videos = VideoLoader.load_videos_from_json(dataset_json_path)
    for video in videos:
        for i, seg in enumerate(video.segments):
            data.append((dataset, video.name, seg.content, seg.start, seg.end))

In [ ]:
import pandas as pd

df = pd.DataFrame(data, columns=["dataset", "video_name", "segment", "start_time", "end_time"])
df.start_time = df.start_time.astype(str).replace("\.", ",", regex=True)
df.end_time = df.end_time.astype(str).replace("\.", ",", regex=True)
df

In [ ]:
df.to_csv("file.csv", encoding="latin1", index=False)